# Lesson 12 - ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ ਨਾਲ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਘਟਾਉਣਾ

ਇਹ ਨੋਟਬੁੱਕ ਦਿਖਾਉਂਦਾ ਹੈ ਕਿ Microsoft Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਲੰਬੀਆਂ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਸੰਦਰਭ (context) ਨੂੰ ਕਿਵੇਂ ਸੰਭਾਲਿਆ ਜਾ ਸਕਦਾ ਹੈ। ਜਿਵੇਂ-ਜਿਵੇਂ ਗੱਲਬਾਤ ਵਧਦੀ ਹੈ, ਟੋਕਨ ਗਿਣਤੀ ਵਧਦੀ ਜਾਂਦੀ ਹੈ — ਜੋ ਅੰਤ ਵਿੱਚ ਮਾਡਲ ਦੀ ਸੰਦਰਭ ਵਿੰਡੋ ਤੋਂ ਵੱਧ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਨੂੰ ਇੱਕ **ਸੰਦਰਭ ਸੰਖੇਪ (context summarization) ਪੈਟਰਨ** ਅਤੇ ਇੱਕ **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ** ਨਾਲ ਪਾਇਦਾਰ ਮੈਮੋਰੀ ਲਈ ਸੰਬੋਧਨ ਕਰਦੇ ਹਾਂ।

## ਤੁਸੀਂ ਕੀ ਸਿੱਖੋਗੇ:
1. **ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਕਿਉਂ ਮਹੱਤਵਪੂਰਣ ਹੈ**: ਟੋਕਨ ਸੀਮਾਵਾਂ ਅਤੇ ਸੰਦਰਭ ਵਿੰਡੋਜ਼ ਦੀ ਸਮਝ
2. **ਸੰਦਰਭ-ਜਾਣੂ ਏਜੰਟ**: ਉਹ ਏਜੰਟ ਬਣਾਉਣਾ ਜੋ ਆਪਣੀ ਗੱਲਬਾਤ ਦਾ ਸੰਦਰਭ ਖੁਦ ਸੰਭਾਲਦੇ ਹਨ
3. **ਸੰਦਰਭ ਸੰਖੇਪ ਪੈਟਰਨ**: ਟੂਲਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਸੰਕੁਚਿਤ ਕਰਨਾ
4. **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ**: ਸਤਾਈ ਮੈਮੋਰੀ ਜੋ ਸੰਦਰਭ ਘਟਾਉਣ ਬਾਵਜੂਦ ਰਹਿੰਦੀ ਹੈ

## ਲੋੜੀਂਦੇ ਗੁਣ:
- Azure OpenAI ਸੈਟਅਪ ਜਿਸ ਵਿੱਚ ਵਾਤਾਵਰਣ ਵੈਰੀਏਬਲ ਸੈੱਟ ਕੀਤੇ ਹੋਣ
- ਪਿਛਲੇ ਪਾਠਾਂ ਤੋਂ ਬੁਨਿਆਦੀ ਏਜੰਟ ਧਾਰਣਾਵਾਂ ਦੀ ਸਮਝ


## ਸੈਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ਕਿਉਂ ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਜ਼ਰੂਰੀ ਹੈ

ਹਰ LLM ਦੀ ਇੱਕ ਸੀਮਿਤ **ਸੰਦਰਭ ਵਿੰਡੋ** ਹੁੰਦੀ ਹੈ — ਇੱਕ ਸੰਘਣੀ ਬੇਨਤੀ ਵਿੱਚ ਉਹ ਜਿੰਨੇ ਟੋਕਨ ਪ੍ਰਕਿਰਿਆਤਮਕ ਕਰ ਸਕਦਾ ਹੈ ਉਸਦੀ ਅਧਿਕਤਮ ਗਿਣਤੀ। ਜਿਵੇਂ-ਜਿਵੇਂ ਬਹੁ-ਚਰਣੀ ਗੱਲਬਾਤ ਅੱਗੇ ਵਧਦੀ ਹੈ:

- ਹਰ ਉਪਭੋਗਤਾ ਸੁਨੇਹੇ ਅਤੇ ਸਹਾਇਕ ਦੇ ਜਵਾਬ ਨਾਲ **ਟੋਕਨ ਗਿਣਤੀ ਰੈਖਿਕ ਤੌਰ ਤੇ ਵੱਧਦੀ ਹੈ**।
- **ਪ੍ਰਾਂਪਟ ਟੋਕਨ ਲਾਗਤ 'ਤੇ ਹਕੀਕਤੀ ਕਬਜ਼ਾ ਕਰਦੇ ਹਨ** ਕਿਉਂਕਿ ਸਾਰਾ ਇਤਿਹਾਸ ਹਰ ਵਾਰੀ ਦੁਬਾਰਾ ਭੇਜਿਆ ਜਾਂਦਾ ਹੈ।
- ਆਖ਼ਿਰਕਾਰ ਗੱਲਬਾਤ **ਸੰਦਰਭ ਵਿੰਡੋ ਤੋਂ ਬਾਹਰ ਚਲੀ ਜਾਂਦੀ ਹੈ** ਅਤੇ ਮਾਡਲ ਜਾਂ ਤਾਂ ਕਟਾਅ ਕਰਦਾ ਹੈ ਜਾਂ ਗਲਤੀ ਦਿਖਾਉਂਦਾ ਹੈ।

### ਸੰਦਰਭ ਪ੍ਰਬੰਧਨ ਲਈ ਰਣਨੀਤੀਆਂ

| ਰਣਨੀਤੀ | ਇਹ ਕਿਵੇਂ ਕੰਮ ਕਰਦੀ ਹੈ | ਤਰਜੀਹ |
|---|---|---|
| **ਕੱਟਣਾ** | ਸਭ ਤੋਂ ਪੁਰਾਣੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਹਟਾਉਣਾ | ਸ਼ੁਰੂਆਤੀ ਸੰਦਰਭ ਖੋ ਜਾਂਦਾ ਹੈ |
| **ਸਾਰ** | ਪੁਰਾਣੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਇਕ ਸਾਰ ਵਿੱਚ ਸੰਕੁਚਿਤ ਕਰਨਾ | ਕੁਝ ਵਿਸਥਾਰ ਖੋ ਜਾਂਦਾ ਹੈ, ਪਰ ਮੁੱਖ ਬਿੰਦੂ ਬਚ ਜਾਂਦੇ ਹਨ |
| **ਸਕ੍ਰੈਚਪੈਡ / ਬਾਹਰੀ ਯਾਦਦਾਸ਼ਤ** | ਗੱਲਬਾਤ ਤੋਂ ਬਾਹਰ ਪ੍ਰਮੁੱਖ ਤੱਥਾਂ ਸੰਭਾਲਣਾ | ਟੂਲ ਕਾਲ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ, ਪਰ ਕਿਸੇ ਵੀ ਘਟਾਅ ਨਾਲ ਜੀਵਿਤ ਰਹਿੰਦੀ ਹੈ |

ਇਸ ਨੋਟਬੁੱਕ ਵਿੱਚ ਅਸੀਂ **ਸਾਰ** ਨੂੰ **ਸਕ੍ਰੈਚਪੈਡ ਟੂਲ** ਨਾਲ ਜੋੜਦੇ ਹਾਂ ਤਾਂ ਜੋ ਏਜੰਟ ਗੱਲਬਾਤ ਦੇ ਇਤਿਹਾਸ ਨੂੰ ਸੰਕੁਚਿਤ ਹੋਣ ਦੌਰਾਨ ਵੀ ਕਾਇਮ ਰੱਖ ਸਕੇ।


## ਸੰਦਰਭ-ਸੂਚਕ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## ਲੰਮੀ ਗੱਲਬਾਤ ਦਾ ਅਨੁਕਰਨ ਕਰਨਾ

ਚਲੋ ਬਹੁ-ਚਰਣੀ ਗੱਲਬਾਤ ਵਿੱਚੋਂ ਲੰਘਦੇ ਹਾਂ ਤਾਂ ਜੋ ਵੇਖਿਆ ਜਾ ਸਕੇ ਕਿ ਸੰਦਰਭ ਕਿਵੇਂ ਇਕੱਠਾ ਹੁੰਦਾ ਹੈ। ਏਜੰਟ ਨੂੰ ਮੁੱਖ ਵੇਰਵਿਆਂ (ਪਸੰਦਾਂ, ਬਜਟ, ਯਾਤਰਾ ਦੀਆਂ ਤਾਰੀਖਾਂ) ਨੂੰ ਚਰਣਾਂ ਵਿੱਚ ਰੱਖਣਾ ਚਾਹੀਦਾ ਹੈ ਅਤੇ ਲਗਾਤਾਰਤਾ ਦਾ ਪ੍ਰਦਰਸ਼ਨ ਕਰਨਾ ਚਾਹੀਦਾ ਹੈ।


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

ਨੋਟ ਕਰੋ ਕਿ ਏਜੰਟ ਪਹਿਲਾਂ ਦੇ ਮੁੜਵੀਂ ਤੋਂ ਸੰਦਰਭ ਕਿਵੇਂ ਸੰਭਾਲਦਾ ਹੈ — ਇਸਨੂੰ ਜਪਾਨ, ਸੁਸ਼ੀ, ਮੰਦਿਰ, ਫੋਟੋਗ੍ਰਾਫੀ, $3000 ਬਜਟ, ਇਕੱਲਾ ਟਰੈਵਲ, ਅਤੇ ਅਪ੍ਰੈਲ ਦੇ ਸਮੇਂ ਬਾਰੇ ਪਤਾ ਹੈ। ਇੱਕ ਛੋਟੀ ਗੱਲਬਾਤ ਵਿੱਚ ਇਹ ਚੰਗੀ ਤਰ੍ਹਾਂ ਕੰਮ ਕਰਦਾ ਹੈ, ਪਰ ਜਿਵੇਂ ਜਿਵੇਂ ਗੱਲਬਾਤ ਵਧਦੀ ਹੈ ਪੂਰਾ ਇਤਿਹਾਸ ਮੁੜ-ਭੇਜਣਾ ਮਹਿੰਗਾ ਹੋ ਜਾਂਦਾ ਹੈ।

ਆਓ ਹੋਰ ਮੁੜਵੀਂ ਨਾਲ ਗੱਲਬਾਤ ਜਾਰੀ ਰੱਖੀਏ ਤਾਂ ਜੋ ਸੰਦਰਭ ਇਕੱਤਰ ਹੋਣ ਨੂੰ ਦੇਖੀਏ:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## ਸੰਦਰਭ ਸਾਰੰਸ਼ੀਕਰਨ ਪੈਟਰਨ

ਜਿਵੇਂ ਹੀ ਗੱਲਬਾਤ ਵਧਦੀ ਹੈ, ਅਸੀਂ ਇੱਕ **ਸਾਰੰਸ਼ੀਕਰਨ ਟੂਲ** ਵਰਤ ਸਕਦੇ ਹਾਂ ਜੋ ਇਕੱਠੇ ਕੀਤੇ ਸੰਦਰਭ ਨੂੰ ਇੱਕ ਸੰਕੁਚਿਤ ਫਾਰਮੈਟ ਵਿੱਚ ਪਰਿਵਰਤਿਤ ਕਰਦਾ ਹੈ। ਏਜੰਟ ਇਹ ਟੂਲ ਕਾਲ ਕਰਦਾ ਹੈ ਤਾਂ ਜੋ ਮੁੱਖ ਪਸੰਦਾਂ ਨੂੰ ਦਰਜ ਕੀਤਾ ਜਾ ਸਕੇ ਤਾਂ ਕਿ ਜੇਕਰ ਪੁਰਾਣੇ ਸੁਨੇਹੇ ਹਟਾ ਦਿੱਤੇ ਜਾਣ, ਤਾਂ ਵੀ ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਬਚੀ ਰਹਿੰਦੀ ਹੈ।

ਇਹ ਪੈਟਰਨ ਵੱਧ ਸੁਖਦਾਇਕ ਇਤਿਹਾਸ ਕਮ ਕਰਨ ਲਈ ਮੂਲ ਪੱਥਰ ਹੈ:
1. ਏਜੰਟ ਗੱਲਬਾਤ ਵਿੱਚੋਂ ਮੁੱਖ ਤੱਥਾਂ ਦੀ ਪਹਚਾਣ ਕਰਦਾ ਹੈ
2. ਇਹਨਾਂ ਨੂੰ ਸਥਿਰ ਕਰਨ ਲਈ ਸਾਰੰਸ਼ੀਕਰਨ ਟੂਲ ਨੂੰ ਕਾਲ ਕਰਦਾ ਹੈ
3. ਪੁਰਾਣੇ ਸੁਨੇਹੇ ਸੁਰੱਖਿਅਤ ਤਰੀਕੇ ਨਾਲ ਹਟਾਏ ਜਾ ਸਕਦੇ ਹਨ ਕਿਉਂਕਿ ਸਾਰੰਸ਼ ਜੋ ਮਹੱਤਵਪੂਰਨ ਹੈ ਉਹ ਕੈਪਚਰ ਕਰਦਾ ਹੈ

ਹੇਠਾਂ ਅਸੀਂ `summarize_preferences` ਟੂਲ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਾਂ ਜੋ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ ਤਾਂ ਜੋ ਉਹ ਜੋ ਕੁਝ ਸਿੱਖਿਆ ਹੈ ਉਸ ਦਾ ਇਕ ਸੰਕੁਚਿਤ ਸਾਰੰਸ਼ ਦਰਜ ਕਰ ਸਕੇ।


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ Microsoft's Agent Framework ਦੀ ਵਰਤੋਂ ਕਰਦਿਆਂ ਲੰਬੇ ਸਮੇਂ ਚੱਲਣ ਵਾਲੀਆਂ ਏਜੰਟ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਸੰਦਰਭ ਨੂੰ ਕਿਵੇਂ ਪ੍ਰਬੰਧਿਤ ਕਰਨਾ ਹੈ, ਇਹ ਸਿੱਖਿਆ:

### ਮੁੱਖ ਧਾਰਣਾਵਾਂ
- **ਸੰਦਰਭ ਵਿੰਡੋਆਂ ਸੀਮਤ ਹੁੰਦੀਆਂ ਹਨ** — ਗੱਲਬਾਤ ਦੇ ਇਤਿਹਾਸ ਵਿੱਚ ਹਰ ਟੋਕਨ ਦੀ ਕੀਮਤ ਹੁੰਦੀ ਹੈ ਅਤੇ ਇਹ ਸੀਮਾ ਵੱਲ ਦੀ ਗਿਣਤੀ ਕਰਦਾ ਹੈ।
- **ਸਾਰਾਂਸ਼ ਯੰਤਰ** ਏਜੰਟ ਨੂੰ ਇਕੱਠੇ ਕੀਤੇ ਸੰਦਰਭ ਨੂੰ ਸੰਕੁਚਿਤ ਸਾਰਾਂਸ਼ਾਂ ਵਿੱਚ ਬਦਲਣ ਦਿੰਦੇ ਹਨ, ਜਿਸ ਨਾਲ ਟੋਕਨ ਦੀ ਵਰਤੋਂ ਘਟਦੀ ਹੈ ਪਰ ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਬਚੀ ਰਹਿੰਦੀ ਹੈ।
- **ਏਜੰਟ ਸਕ੍ਰੈਚਪੈਡ** ਸਥਾਈ ਬਾਹਰੀ ਯਾਦ ਦਿੰਦੇ ਹਨ ਜੋ ਕਿਸੇ ਵੀ ਗੱਲਬਾਤ ਵਿੱਚ ਕਮੀ ਤੋਂ ਬਾਅਦ ਵੀ ਟਿਕਿਆ ਰਹਿੰਦਾ ਹੈ।

### ਤੁਸੀਂ ਕੀ ਬਣਾਇਆ
- ਇੱਕ **ਸੰਦਰਭ-ਜਾਣਕਾਰ ਏਜੰਟ** ਜੋ ਬਹੁ-ਚਰ ਗੱਲਬਾਤਾਂ ਵਿੱਚ ਲਗਾਤਾਰਤਾ ਕਾਇਮ ਰੱਖਦਾ ਹੈ
- ਇੱਕ **ਸਾਰਾਂਸ਼ ਯੰਤਰ** (`summarize_preferences`) ਜੋ ਮੁੱਖ ਉਪਭੋਗਤਾ ਵੇਰਵੇ ਇੱਕ ਸੰਕੁਚਿਤ ਰੂਪ ਵਿੱਚ ਦਰਜ ਕਰਦਾ ਹੈ
- ਇੱਕ **ਬਹੁ-ਚਰ ਗੱਲਬਾਤ** ਜੋ ਸੰਦਰਭ ਨੂੰ ਸੰਭਾਲਣ ਅਤੇ ਬਦਲਾਅ ਦਾ ਪ੍ਰਦਰਸ਼ਨ ਕਰਦੀ ਹੈ

### ਅਸਲ ਦੁਨੀਆ ਦੇ ਅਰਜ਼ੀਕਾਰ
- **ਗਾਹਕ ਸੇਵਾ ਬੋਟਸ**: ਲੰਬੀਆਂ ਸਹਾਇਤਾ ਸੈਸ਼ਨਾਂ ਵਿੱਚ ਪਸੰਦਾਂ ਨੂੰ ਯਾਦ ਰੱਖੋ
- **ਨਿੱਜੀ ਸਹਾਇਕ**: ਬਿਨਾਂ ਸੰਦਰਭ ਦੁਬਾਰਾ ਸਮਝਾਏ ਚੱਲ ਰਹੇ ਪ੍ਰਾਜੈਕਟਾਂ ਨੂੰ ਟਰੈਕ ਕਰੋ
- **ਅਧਿਆਪਕ ਟਿਊਟਰ**: ਬਹੁਤ ਸਾਰੀਆਂ ਬਾਤਾਂ ਵਿੱਚ ਵਿਦਿਆਰਥੀ ਦੀ ਪ੍ਰਗਤੀ ਨੂੰ ਕਾਇਮ ਰੱਖੋ

### ਅੱਗੇ ਦੇ ਕਦਮ
- ਫਾਈਲ-ਆਧਾਰਿਤ ਸਥਾਇਤਵਾਲੇ ਪੂਰੇ ਸਕ੍ਰੈਚਪੈਡ ਯੰਤਰ ਦਾ ਕਾਰਜਾਨਵয়ন ਕਰੋ
- ਸਾਰਾਂਸ਼ ਬਣਾਉਣ ਤੋਂ ਬਾਅਦ ਆਪਣੇ ਆਪ ਇਤਿਹਾਸ ਕੱਟਣਾ ਸ਼ੁਰੂ ਕਰੋ
- ਸੈਮਾਂਟਿਕ ਯਾਦ ਲਈ ਵੈਕਟਰ ਡੇਟਾਬੇਸ ਨਾਲ ਜੋੜੋ
- ਐਸੇ ਏਜੰਟ ਬਣਾਓ ਜੋ ਪੂਰੇ ਸੰਦਰਭ ਨਾਲ ਦਿਨਾਂ ਬਾਅਦ ਗੱਲਬਾਤ ਜਾਰੀ ਰੱਖ ਸਕਣ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋ ਜਾਂਚ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਜਾਣੋ ਕਿ ਆਟੋਮੈਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੇ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਨ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਤੋਂ ਉਪਜੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫ਼ਹਮੀ ਜਾਂ ਗਲਤ ਵੀਖਿਆਵ ਦੀ ਜ਼ਿੰਮੇਵਾਰੀ ਨਹੀਂ ਲੈਂਦੇ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
